# BERT pretraining and toxic-comment fine-tuning

This notebook builds a compact BERT encoder from scratch for the Jigsaw toxic-comment corpus.

The workflow is intentionally split into two stages:

1. **Self-supervised pretraining** on the training-text split only:
   - WordPiece tokenizer trained from scratch.
   - Masked Language Modeling (MLM) with dynamic 15% masking.
   - Next Sentence Prediction (NSP) with balanced positive and negative pairs.
2. **Supervised fine-tuning**:
   - Transfer the pretrained BERT encoder.
   - Add a two-class classification head.
   - Fine-tune on the original non-toxic/toxic labels.
   - Select the best checkpoint by validation macro-F1 and evaluate once on the test set.

Validation and test text never updates the tokenizer or pretrained weights, preventing data leakage.


## 1. Imports and reproducibility


In [3]:
import gc
import json
import math
import random
import re
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from tokenizers import Tokenizer
from tokenizers.decoders import WordPiece as WordPieceDecoder
from tokenizers.models import WordPiece
from tokenizers.normalizers import BertNormalizer
from tokenizers.pre_tokenizers import BertPreTokenizer
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordPieceTrainer

from transformers import (
    BertConfig,
    BertForPreTraining,
    BertForSequenceClassification,
    PreTrainedTokenizerFast,
    get_linear_schedule_with_warmup,
)


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BERT_ROOT = PROJECT_ROOT / "model" / "bert_tuning"
PRETRAINED_DIR = BERT_ROOT / "pretrained"
CLASSIFIER_DIR = BERT_ROOT / "toxic_classifier"
BERT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Device:", device)


Project root: /Users/andreyvargassolis/Desktop/Data_Science/Transformer_CommentClassification
Device: mps


## 2. Load and split the Jigsaw dataset

All six original toxicity labels are collapsed into one binary label. Splitting happens before tokenizer training or self-supervised pretraining so validation and test comments remain unseen.


In [5]:
DATA_SOURCE = (
    "hf://datasets/thesofakillers/"
    "jigsaw-toxic-comment-classification-challenge/train.csv"
)
LABEL_COLUMNS = [
    "toxic",
    "severe_toxic",
    "insult",
    "identity_hate",
    "obscene",
    "threat",
]

df = pl.read_csv(DATA_SOURCE)

df_model = (
    df.with_columns(
        pl.when(
            pl.any_horizontal([pl.col(column) == 1 for column in LABEL_COLUMNS])
        )
        .then(1)
        .otherwise(0)
        .alias("label")
    )
    .select(["id", "comment_text", "label"])
    .drop_nulls(["comment_text", "label"])
    .filter(pl.col("comment_text").str.strip_chars().str.len_chars() > 0)
)

texts = df_model["comment_text"].to_list()
labels = df_model["label"].to_list()

X_train, X_temp, y_train, y_temp = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=SEED,
    stratify=labels,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print(f"Clean rows: {len(df_model):,}")
print(f"Training:   {len(X_train):,}")
print(f"Validation: {len(X_val):,}")
print(f"Test:       {len(X_test):,}")
print("Training class counts:", np.bincount(y_train, minlength=2))


Clean rows: 159,571
Training:   127,656
Validation: 15,957
Test:       15,958
Training class counts: [114676  12980]


## 3. Train a BERT WordPiece tokenizer

BERT needs five special tokens:

- `[PAD]`: fixed-length padding
- `[UNK]`: unknown subword
- `[CLS]`: pooled sequence representation
- `[SEP]`: sequence separator
- `[MASK]`: MLM replacement token

For paired inputs, token type `0` identifies sentence A and token type `1` identifies sentence B.


In [6]:
VOCAB_SIZE = 30_000
MAX_LEN = 128
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

backend_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
backend_tokenizer.normalizer = BertNormalizer(
    clean_text=True,
    handle_chinese_chars=True,
    strip_accents=None,
    lowercase=True,
)
backend_tokenizer.pre_tokenizer = BertPreTokenizer()

wordpiece_trainer = WordPieceTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=SPECIAL_TOKENS,
    continuing_subword_prefix="##",
)

backend_tokenizer.train_from_iterator(
    X_train,
    trainer=wordpiece_trainer,
    length=len(X_train),
)

cls_id = backend_tokenizer.token_to_id("[CLS]")
sep_id = backend_tokenizer.token_to_id("[SEP]")

backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_id), ("[SEP]", sep_id)],
)
backend_tokenizer.decoder = WordPieceDecoder(prefix="##")

tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend_tokenizer,
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    model_max_length=MAX_LEN,
)

tokenizer.save_pretrained(PRETRAINED_DIR)

print("Vocabulary size:", len(tokenizer))
print("Special token IDs:", {
    token: tokenizer.convert_tokens_to_ids(token)
    for token in SPECIAL_TOKENS
})
print("Tokenizer saved to:", PRETRAINED_DIR)





Vocabulary size: 30000
Special token IDs: {'[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4}
Tokenizer saved to: /Users/andreyvargassolis/Desktop/Data_Science/Transformer_CommentClassification/model/bert_tuning/pretrained


In [7]:
pair_example = tokenizer(
    "The first sentence belongs to segment A.",
    "The second sentence belongs to segment B.",
    max_length=MAX_LEN,
    truncation="longest_first",
    padding="max_length",
    return_attention_mask=True,
    return_token_type_ids=True,
    return_special_tokens_mask=True,
)

active_length = sum(pair_example["attention_mask"])
print("Tokens:", tokenizer.convert_ids_to_tokens(pair_example["input_ids"][:active_length]))
print("Token type IDs:", pair_example["token_type_ids"][:active_length])
print("Special-token mask:", pair_example["special_tokens_mask"][:active_length])

assert pair_example["input_ids"][0] == tokenizer.cls_token_id
assert pair_example["input_ids"][active_length - 1] == tokenizer.sep_token_id
assert 0 in pair_example["token_type_ids"][:active_length]
assert 1 in pair_example["token_type_ids"][:active_length]


Tokens: ['[CLS]', 'the', 'first', 'sentence', 'belongs', 'to', 'segment', 'a', '.', '[SEP]', 'the', 'second', 'sentence', 'belongs', 'to', 'segment', 'b', '.', '[SEP]']
Token type IDs: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Special-token mask: [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1]


## 4. Build balanced NSP examples

A positive NSP example contains adjacent segments from the same comment and receives label `0` (`IsNext`). A negative example pairs the same first segment with a segment sampled from a different comment and receives label `1` (`NotNext`).

The dataset exposes one positive and one negative item per anchor, producing exact class balance. Short comments without two usable sentences are split into two word chunks when possible.


In [8]:
SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?])\s+|\n+")
MIN_SEGMENT_WORDS = 3


def split_comment_into_segments(text, min_words=MIN_SEGMENT_WORDS):
    normalized = re.sub(r"\s+", " ", str(text)).strip()
    if not normalized:
        return []

    segments = [
        segment.strip()
        for segment in SENTENCE_BOUNDARY.split(normalized)
        if len(segment.strip().split()) >= min_words
    ]
    if len(segments) >= 2:
        return segments

    words = normalized.split()
    if len(words) >= 2 * min_words:
        midpoint = len(words) // 2
        return [" ".join(words[:midpoint]), " ".join(words[midpoint:])]

    return []


class BertNSPDataset(Dataset):
    """Balanced positive/negative sentence pairs for BERT pretraining."""

    def __init__(
        self,
        comments,
        tokenizer,
        max_length=128,
        seed=42,
        max_anchors=None,
    ):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.seed = seed
        self.documents = [
            segments
            for text in comments
            if len(segments := split_comment_into_segments(text)) >= 2
        ]

        if len(self.documents) < 2:
            raise ValueError("NSP requires at least two comments with two usable segments.")

        self.anchors = [
            (document_index, segment_index)
            for document_index, segments in enumerate(self.documents)
            for segment_index in range(len(segments) - 1)
        ]

        if max_anchors is not None and len(self.anchors) > max_anchors:
            rng = random.Random(seed)
            self.anchors = rng.sample(self.anchors, max_anchors)

    def __len__(self):
        return 2 * len(self.anchors)

    def __getitem__(self, index):
        anchor_index = index // 2
        is_positive = index % 2 == 0
        document_index, segment_index = self.anchors[anchor_index]

        text_a = self.documents[document_index][segment_index]

        if is_positive:
            text_b = self.documents[document_index][segment_index + 1]
            next_sentence_label = 0
        else:
            rng = random.Random(self.seed + index)
            negative_document_index = rng.randrange(len(self.documents) - 1)
            if negative_document_index >= document_index:
                negative_document_index += 1
            negative_document = self.documents[negative_document_index]
            text_b = negative_document[rng.randrange(len(negative_document))]
            next_sentence_label = 1

        encoded = self.tokenizer(
            text_a,
            text_b,
            add_special_tokens=True,
            max_length=self.max_length,
            truncation="longest_first",
            padding="max_length",
            return_attention_mask=True,
            return_token_type_ids=True,
            return_special_tokens_mask=True,
        )
        encoded["next_sentence_label"] = next_sentence_label
        return encoded


In [9]:
# Set MAX_PRETRAIN_ANCHORS to an integer for a short experiment.
# Keep it as None to use every available training anchor.
MAX_PRETRAIN_ANCHORS = None
MAX_VALIDATION_ANCHORS = 10_000

pretrain_dataset = BertNSPDataset(
    X_train,
    tokenizer,
    max_length=MAX_LEN,
    seed=SEED,
    max_anchors=MAX_PRETRAIN_ANCHORS,
)
pretrain_val_dataset = BertNSPDataset(
    X_val,
    tokenizer,
    max_length=MAX_LEN,
    seed=SEED + 1,
    max_anchors=MAX_VALIDATION_ANCHORS,
)

print(f"Training NSP pairs:   {len(pretrain_dataset):,}")
print(f"Validation NSP pairs: {len(pretrain_val_dataset):,}")
print("NSP balance is exactly 50% IsNext and 50% NotNext.")


Training NSP pairs:   807,040
Validation NSP pairs: 20,000
NSP balance is exactly 50% IsNext and 50% NotNext.


## 5. Dynamic MLM masking and pretraining DataLoaders

Masking is created at batch time, so a sequence can receive different masks in different epochs. Exactly 15% of eligible tokens are selected on average:

- 80% are replaced by `[MASK]`
- 10% are replaced by a random vocabulary token
- 10% remain unchanged

`[CLS]`, `[SEP]`, `[PAD]`, `[MASK]`, and other special tokens are never selected as MLM targets. MLM labels use `-100` at unmasked positions, which PyTorch cross-entropy ignores.


In [10]:
class DynamicMLMAndNSPCollator:
    def __init__(self, tokenizer, mlm_probability=0.15):
        if tokenizer.mask_token_id is None:
            raise ValueError("The tokenizer must define a [MASK] token.")
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability

    def __call__(self, examples):
        batch = {
            key: torch.tensor([example[key] for example in examples], dtype=torch.long)
            for key in (
                "input_ids",
                "attention_mask",
                "token_type_ids",
                "special_tokens_mask",
                "next_sentence_label",
            )
        }

        special_tokens_mask = batch.pop("special_tokens_mask").bool()
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"].bool()
        labels = input_ids.clone()

        eligible = attention_mask & ~special_tokens_mask
        probability_matrix = torch.full(
            labels.shape,
            self.mlm_probability,
            dtype=torch.float32,
        )
        masked_positions = torch.bernoulli(probability_matrix).bool() & eligible

        # Guarantee at least one MLM target for every non-empty sequence.
        for row in range(masked_positions.size(0)):
            if not masked_positions[row].any():
                candidates = torch.nonzero(eligible[row], as_tuple=False).flatten()
                if candidates.numel() > 0:
                    selected = candidates[torch.randint(candidates.numel(), (1,))]
                    masked_positions[row, selected] = True

        labels[~masked_positions] = -100

        # 80% of selected tokens become [MASK].
        replaced = (
            torch.bernoulli(torch.full(labels.shape, 0.80)).bool()
            & masked_positions
        )
        input_ids[replaced] = self.tokenizer.mask_token_id

        # Half of the remaining 20% become random tokens: 10% overall.
        random_replaced = (
            torch.bernoulli(torch.full(labels.shape, 0.50)).bool()
            & masked_positions
            & ~replaced
        )
        random_tokens = torch.randint(
            low=0,
            high=len(self.tokenizer),
            size=labels.shape,
            dtype=torch.long,
        )
        input_ids[random_replaced] = random_tokens[random_replaced]

        batch["labels"] = labels
        return batch


In [11]:
PRETRAIN_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
MLM_PROBABILITY = 0.15
NUM_WORKERS = 0  # Reliable inside Jupyter on macOS.

pretrain_collator = DynamicMLMAndNSPCollator(
    tokenizer,
    mlm_probability=MLM_PROBABILITY,
)

loader_generator = torch.Generator().manual_seed(SEED)

pretrain_loader = DataLoader(
    pretrain_dataset,
    batch_size=PRETRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=pretrain_collator,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    generator=loader_generator,
)
pretrain_val_loader = DataLoader(
    pretrain_val_dataset,
    batch_size=PRETRAIN_BATCH_SIZE,
    shuffle=False,
    collate_fn=pretrain_collator,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

pretraining_batch = next(iter(pretrain_loader))
masked_positions = pretraining_batch["labels"] != -100

print({key: tuple(value.shape) for key, value in pretraining_batch.items()})
print("MLM targets in batch:", masked_positions.sum().item())
print(
    "NSP labels:",
    torch.bincount(
        pretraining_batch["next_sentence_label"],
        minlength=2,
    ).tolist(),
)

assert pretraining_batch["input_ids"].shape[1] == MAX_LEN
assert masked_positions.any(dim=1).all()
assert set(pretraining_batch["next_sentence_label"].tolist()).issubset({0, 1})


{'input_ids': (16, 128), 'attention_mask': (16, 128), 'token_type_ids': (16, 128), 'next_sentence_label': (16,), 'labels': (16, 128)}
MLM targets in batch: 84
NSP labels: [9, 7]


## 6. Create the BERT encoder and pretraining heads

`BertForPreTraining` contains one shared BERT encoder plus the MLM vocabulary head and the binary NSP head. This compact configuration mirrors the scale of the earlier custom Transformer while using BERT components: learned position embeddings, token type embeddings, GELU activations, residual connections, and layer normalization.


In [12]:
bert_config = BertConfig(
    vocab_size=len(tokenizer),
    hidden_size=256,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=1024,
    hidden_act="gelu",
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    max_position_embeddings=MAX_LEN,
    type_vocab_size=2,
    initializer_range=0.02,
    layer_norm_eps=1e-12,
    pad_token_id=tokenizer.pad_token_id,
    position_embedding_type="absolute",
)

pretraining_model = BertForPreTraining(bert_config).to(device)

total_parameters = sum(parameter.numel() for parameter in pretraining_model.parameters())
trainable_parameters = sum(
    parameter.numel()
    for parameter in pretraining_model.parameters()
    if parameter.requires_grad
)

print(pretraining_model)
print(f"Total parameters:     {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")


BertForPreTraining(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30000, 256, padding_idx=0)
      (position_embeddings): Embedding(128, 256)
      (token_type_embeddings): Embedding(2, 256)
      (LayerNorm): LayerNorm((256,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=256, out_features=256, bias=True)
              (key): Linear(in_features=256, out_features=256, bias=True)
              (value): Linear(in_features=256, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=256, out_features=256, bias=True)
              (LayerNorm): LayerNorm((256,), eps=1e-12, e

In [13]:
# One forward pass verifies MLM labels, NSP labels, masks, and token types.
sample_batch = {
    key: value.to(device)
    for key, value in pretraining_batch.items()
}

pretraining_model.eval()
with torch.no_grad():
    sample_output = pretraining_model(**sample_batch)

print("Combined MLM + NSP loss:", float(sample_output.loss))
print("MLM logits:", tuple(sample_output.prediction_logits.shape))
print("NSP logits:", tuple(sample_output.seq_relationship_logits.shape))

assert sample_output.prediction_logits.shape == (
    pretraining_batch["input_ids"].shape[0],
    MAX_LEN,
    len(tokenizer),
)
assert sample_output.seq_relationship_logits.shape == (
    pretraining_batch["input_ids"].shape[0],
    2,
)


Combined MLM + NSP loss: 11.051352500915527
MLM logits: (16, 128, 30000)
NSP logits: (16, 2)


## 7. MLM + NSP pretraining loop

The reported MLM accuracy is calculated only at masked target positions. NSP accuracy is calculated across every sentence pair. The latest completed epoch is saved to `model/bert_tuning/pretrained`, making long training runs recoverable at epoch boundaries.


In [14]:
def run_pretraining_epoch(
    model,
    data_loader,
    device,
    optimizer=None,
    scheduler=None,
    gradient_accumulation_steps=1,
):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    total_batches = 0
    mlm_correct = 0
    mlm_total = 0
    nsp_correct = 0
    nsp_total = 0

    if is_training:
        optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        data_loader,
        desc="Pretraining" if is_training else "Pretraining validation",
    )

    for step, batch in enumerate(progress, start=1):
        batch = {key: value.to(device) for key, value in batch.items()}

        with torch.set_grad_enabled(is_training):
            output = model(**batch)
            loss = output.loss

            if is_training:
                (loss / gradient_accumulation_steps).backward()

        should_update = (
            step % gradient_accumulation_steps == 0
            or step == len(data_loader)
        )
        if is_training and should_update:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            valid_mlm = batch["labels"] != -100
            mlm_predictions = output.prediction_logits.argmax(dim=-1)
            mlm_correct += (
                mlm_predictions[valid_mlm] == batch["labels"][valid_mlm]
            ).sum().item()
            mlm_total += valid_mlm.sum().item()

            nsp_predictions = output.seq_relationship_logits.argmax(dim=-1)
            nsp_correct += (
                nsp_predictions == batch["next_sentence_label"]
            ).sum().item()
            nsp_total += batch["next_sentence_label"].numel()

        total_loss += loss.item()
        total_batches += 1
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return {
        "loss": total_loss / max(total_batches, 1),
        "mlm_accuracy": mlm_correct / max(mlm_total, 1),
        "nsp_accuracy": nsp_correct / max(nsp_total, 1),
    }


In [15]:
PRETRAIN_EPOCHS = 5
PRETRAIN_LR = 5e-4
PRETRAIN_WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

pretrain_optimizer = torch.optim.AdamW(
    pretraining_model.parameters(),
    lr=PRETRAIN_LR,
    weight_decay=PRETRAIN_WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(pretrain_loader) / GRADIENT_ACCUMULATION_STEPS
)
total_pretrain_steps = PRETRAIN_EPOCHS * updates_per_epoch
warmup_steps = int(WARMUP_RATIO * total_pretrain_steps)

pretrain_scheduler = get_linear_schedule_with_warmup(
    pretrain_optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_pretrain_steps,
)

pretraining_history = []

for epoch in range(PRETRAIN_EPOCHS):
    print(f"\nPretraining epoch {epoch + 1}/{PRETRAIN_EPOCHS}")
    print("-" * 60)

    train_metrics = run_pretraining_epoch(
        pretraining_model,
        pretrain_loader,
        device,
        optimizer=pretrain_optimizer,
        scheduler=pretrain_scheduler,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    )
    validation_metrics = run_pretraining_epoch(
        pretraining_model,
        pretrain_val_loader,
        device,
    )

    epoch_metrics = {
        "epoch": epoch + 1,
        "train": train_metrics,
        "validation": validation_metrics,
    }
    pretraining_history.append(epoch_metrics)

    print("Train:", train_metrics)
    print("Validation:", validation_metrics)

    # Save the latest fully completed epoch.
    pretraining_model.save_pretrained(PRETRAINED_DIR)
    tokenizer.save_pretrained(PRETRAINED_DIR)
    (PRETRAINED_DIR / "pretraining_history.json").write_text(
        json.dumps(pretraining_history, indent=2) + "\n"
    )

print("Pretrained BERT saved to:", PRETRAINED_DIR)



Pretraining epoch 1/5
------------------------------------------------------------


Pretraining:   0%|          | 0/50440 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Prepare supervised toxic-comment DataLoaders

Fine-tuning uses single-sequence BERT inputs:

`[CLS] comment tokens [SEP] [PAD] ...`

The attention mask excludes padding. Token type IDs are all zero because there is only one segment. The train, validation, and test splits are exactly the ones created before pretraining.


In [ ]:
class ToxicCommentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index],
            add_special_tokens=True,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_attention_mask=True,
            return_token_type_ids=True,
        )
        return {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(
                encoded["attention_mask"],
                dtype=torch.long,
            ),
            "token_type_ids": torch.tensor(
                encoded["token_type_ids"],
                dtype=torch.long,
            ),
            "labels": torch.tensor(self.labels[index], dtype=torch.long),
        }


FINE_TUNE_BATCH_SIZE = 32

train_dataset = ToxicCommentDataset(
    X_train, y_train, tokenizer, max_length=MAX_LEN
)
val_dataset = ToxicCommentDataset(
    X_val, y_val, tokenizer, max_length=MAX_LEN
)
test_dataset = ToxicCommentDataset(
    X_test, y_test, tokenizer, max_length=MAX_LEN
)

fine_tune_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=FINE_TUNE_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
    generator=fine_tune_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=FINE_TUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    test_dataset,
    batch_size=FINE_TUNE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

classification_batch = next(iter(train_loader))
print({key: tuple(value.shape) for key, value in classification_batch.items()})
assert classification_batch["input_ids"].shape[1] == MAX_LEN


## 9. Transfer the pretrained encoder into a classifier

Only the MLM and NSP heads are discarded. Every BERT encoder weight—including token, position, and segment embeddings—is transferred to the sequence-classification model. A new randomly initialized two-class head is then fine-tuned together with the encoder.


In [ ]:
# Release the in-memory training model before loading the saved checkpoint.
del sample_output, sample_batch, pretraining_batch
del pretraining_model
gc.collect()
if device.type == "mps":
    torch.mps.empty_cache()
elif device.type == "cuda":
    torch.cuda.empty_cache()

saved_pretraining_model = BertForPreTraining.from_pretrained(PRETRAINED_DIR)

classification_config = BertConfig.from_pretrained(
    PRETRAINED_DIR,
    num_labels=2,
    id2label={0: "NON_TOXIC", 1: "TOXIC"},
    label2id={"NON_TOXIC": 0, "TOXIC": 1},
)

classification_model = BertForSequenceClassification(
    classification_config
)

transfer_result = classification_model.bert.load_state_dict(
    saved_pretraining_model.bert.state_dict(),
    strict=True,
)

assert not transfer_result.missing_keys
assert not transfer_result.unexpected_keys

del saved_pretraining_model
gc.collect()

classification_model = classification_model.to(device)
print("Transferred the complete pretrained BERT encoder.")
print(
    "Classification parameters:",
    f"{sum(parameter.numel() for parameter in classification_model.parameters()):,}",
)


## 10. Fine-tune for toxic-comment classification

Inverse-frequency class weights compensate for the strong class imbalance. The best checkpoint is chosen by validation macro-F1 rather than raw accuracy so performance on the toxic minority class contributes equally.


In [ ]:
class_counts = np.bincount(y_train, minlength=2)
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device,
)

classification_criterion = nn.CrossEntropyLoss(weight=class_weights)

FINE_TUNE_EPOCHS = 4
FINE_TUNE_LR = 2e-5
FINE_TUNE_WEIGHT_DECAY = 0.01
FINE_TUNE_WARMUP_RATIO = 0.10

fine_tune_optimizer = torch.optim.AdamW(
    classification_model.parameters(),
    lr=FINE_TUNE_LR,
    weight_decay=FINE_TUNE_WEIGHT_DECAY,
)

total_fine_tune_steps = FINE_TUNE_EPOCHS * len(train_loader)
fine_tune_scheduler = get_linear_schedule_with_warmup(
    fine_tune_optimizer,
    num_warmup_steps=int(
        FINE_TUNE_WARMUP_RATIO * total_fine_tune_steps
    ),
    num_training_steps=total_fine_tune_steps,
)

print("Class counts:", class_counts)
print("Class weights:", class_weights)


In [ ]:
def run_classification_epoch(
    model,
    data_loader,
    criterion,
    device,
    optimizer=None,
    scheduler=None,
):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    predictions = []
    targets = []

    progress = tqdm(
        data_loader,
        desc="Fine-tuning" if is_training else "Classification evaluation",
    )

    for batch in progress:
        labels = batch["labels"].to(device)
        model_inputs = {
            key: value.to(device)
            for key, value in batch.items()
            if key != "labels"
        }

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            output = model(**model_inputs)
            loss = criterion(output.logits, labels)

            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                )
                optimizer.step()
                scheduler.step()

        batch_predictions = output.logits.argmax(dim=-1)
        predictions.extend(batch_predictions.detach().cpu().tolist())
        targets.extend(labels.detach().cpu().tolist())
        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    return {
        "loss": total_loss / max(len(data_loader), 1),
        "accuracy": float((predictions == targets).mean()),
        "macro_f1": f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "predictions": predictions,
        "targets": targets,
    }


In [ ]:
fine_tuning_history = []
best_validation_f1 = -1.0

for epoch in range(FINE_TUNE_EPOCHS):
    print(f"\nFine-tuning epoch {epoch + 1}/{FINE_TUNE_EPOCHS}")
    print("-" * 60)

    train_metrics = run_classification_epoch(
        classification_model,
        train_loader,
        classification_criterion,
        device,
        optimizer=fine_tune_optimizer,
        scheduler=fine_tune_scheduler,
    )
    validation_metrics = run_classification_epoch(
        classification_model,
        val_loader,
        classification_criterion,
        device,
    )

    history_entry = {
        "epoch": epoch + 1,
        "train": {
            key: value
            for key, value in train_metrics.items()
            if key not in {"predictions", "targets"}
        },
        "validation": {
            key: value
            for key, value in validation_metrics.items()
            if key not in {"predictions", "targets"}
        },
    }
    fine_tuning_history.append(history_entry)

    print("Train:", history_entry["train"])
    print("Validation:", history_entry["validation"])

    if validation_metrics["macro_f1"] > best_validation_f1:
        best_validation_f1 = validation_metrics["macro_f1"]
        classification_model.save_pretrained(CLASSIFIER_DIR)
        tokenizer.save_pretrained(CLASSIFIER_DIR)
        print(
            "Saved new best classifier with validation macro-F1:",
            f"{best_validation_f1:.4f}",
        )

    (CLASSIFIER_DIR / "fine_tuning_history.json").write_text(
        json.dumps(fine_tuning_history, indent=2) + "\n"
    )

print("Best classifier saved to:", CLASSIFIER_DIR)


## 11. Final test evaluation

The test split is evaluated only after model selection. This cell reloads the best validation checkpoint and reports class-level precision, recall, F1, and the confusion matrix.


In [ ]:
best_classifier = BertForSequenceClassification.from_pretrained(
    CLASSIFIER_DIR
).to(device)

test_metrics = run_classification_epoch(
    best_classifier,
    test_loader,
    classification_criterion,
    device,
)

print(f"Test loss:     {test_metrics['loss']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test macro-F1: {test_metrics['macro_f1']:.4f}")
print()
print(
    classification_report(
        test_metrics["targets"],
        test_metrics["predictions"],
        labels=[0, 1],
        target_names=["Non-toxic", "Toxic"],
        digits=4,
        zero_division=0,
    )
)

test_confusion_matrix = confusion_matrix(
    test_metrics["targets"],
    test_metrics["predictions"],
    labels=[0, 1],
)
print("Confusion matrix:")
print(test_confusion_matrix)

test_summary = {
    "loss": test_metrics["loss"],
    "accuracy": test_metrics["accuracy"],
    "macro_f1": test_metrics["macro_f1"],
    "classification_report": classification_report(
        test_metrics["targets"],
        test_metrics["predictions"],
        labels=[0, 1],
        target_names=["Non-toxic", "Toxic"],
        output_dict=True,
        zero_division=0,
    ),
    "confusion_matrix": test_confusion_matrix.tolist(),
}
(CLASSIFIER_DIR / "test_metrics.json").write_text(
    json.dumps(test_summary, indent=2) + "\n"
)


## 12. Inference with the fine-tuned BERT classifier


In [ ]:
def predict_comment(
    text,
    model=best_classifier,
    tokenizer=tokenizer,
    device=device,
):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("The comment cannot be empty.")

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
        return_attention_mask=True,
        return_token_type_ids=True,
    )
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    model.eval()
    with torch.inference_mode():
        logits = model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_class = probabilities.argmax().item()
    return {
        "label_id": predicted_class,
        "label": model.config.id2label[predicted_class],
        "non_toxic_probability": probabilities[0].item(),
        "toxic_probability": probabilities[1].item(),
    }


examples = [
    "Thank you for explaining this so clearly.",
    "You are such an idiot.",
]

for example in examples:
    print(example)
    print(predict_comment(example))
    print()


## Training artifacts

After all training cells complete:

```text
model/bert_tuning/
├── pretrained/
│   ├── config.json
│   ├── model.safetensors
│   ├── tokenizer.json
│   ├── tokenizer_config.json
│   └── pretraining_history.json
└── toxic_classifier/
    ├── config.json
    ├── model.safetensors
    ├── tokenizer.json
    ├── tokenizer_config.json
    ├── fine_tuning_history.json
    └── test_metrics.json
```

The original custom Transformer artifacts in `model/` remain unchanged.
